In [21]:
# ACC102 Track 2: US Stock Return Analysis (2024)
# Author: Jingyi Guan | Student ID: 2473524
# Data Source: WRDS CRSP | Date Accessed: 2024-04-24

# 1. Setup & Connect to WRDS


import wrds
import pandas as pd
df = pd.read_csv("stock_2022_data.csv")


try:
    df_all = pd.read_csv("stock_2022_data.csv")
    print("Data loaded from local CSV successfully.")

except FileNotFoundError:
    print("Local CSV not found, connecting to WRDS database...")
    db = wrds.Connection()
    
username = "jingyiguan"
db = wrds.Connection(wrds_username=username)

Data loaded from local CSV successfully.
Loading library list...
Done


In [22]:
# 2. Define Parameters

ticker_list = ['NVDA', 'MSFT', "AAPL"]
start_date = "2024-01-01"
end_date = "2024-12-31"

# 3. SQL Query
selected_columns = """
b.htsymbol AS ticker,
a.date,
a.permno,
a.prc,
a.ret
"""

ticker_str = "', '".join(ticker_list)


sql_query = f"""
SELECT {selected_columns}
FROM crsp.msf AS a
LEFT JOIN crsp.msfhdr AS b
    ON a.permno = b.permno
WHERE b.htsymbol IN ('{ticker_str}')
AND a.date >= '{start_date}'
AND a.date <= '{end_date}'
"""
print(sql_query)

# 4. Load & Clean Data
df = db.raw_sql(sql_query)
print(df.head())


SELECT 
b.htsymbol AS ticker,
a.date,
a.permno,
a.prc,
a.ret

FROM crsp.msf AS a
LEFT JOIN crsp.msfhdr AS b
    ON a.permno = b.permno
WHERE b.htsymbol IN ('NVDA', 'MSFT', 'AAPL')
AND a.date >= '2024-01-01'
AND a.date <= '2024-12-31'

  ticker        date  permno        prc       ret
0   MSFT  2024-01-31   10107  397.57999  0.057281
1   MSFT  2024-02-29   10107  413.64001  0.042281
2   MSFT  2024-03-28   10107     420.72  0.017116
3   MSFT  2024-04-30   10107  389.32999  -0.07461
4   MSFT  2024-05-31   10107     415.13  0.068194


In [23]:
stock_joined = db.raw_sql(sql_query, date_cols=["date"])
stock_joined

,ticker,date,permno,prc,ret
0,MSFT,2024-01-31,10107,397.57999,0.057281
1,MSFT,2024-02-29,10107,413.64001,0.042281
2,MSFT,2024-03-28,10107,420.72,0.017116
3,MSFT,2024-04-30,10107,389.32999,-0.07461
4,MSFT,2024-05-31,10107,415.13,0.068194
5,MSFT,2024-06-28,10107,446.95001,0.076651
6,MSFT,2024-07-31,10107,418.35001,-0.063989
7,MSFT,2024-08-30,10107,417.14001,-0.0011
8,MSFT,2024-09-30,10107,430.29999,0.031548
9,MSFT,2024-10-31,10107,406.35001,-0.055659


In [24]:
# Rename columns
stock_joined_renamed = stock_joined.rename(columns={
    "prc": "price",
    "ret": "monthly_return"
})

stock_joined_renamed.head()


,ticker,date,permno,price,monthly_return
0,MSFT,2024-01-31,10107,397.57999,0.057281
1,MSFT,2024-02-29,10107,413.64001,0.042281
2,MSFT,2024-03-28,10107,420.72,0.017116
3,MSFT,2024-04-30,10107,389.32999,-0.07461
4,MSFT,2024-05-31,10107,415.13,0.068194


In [25]:
# Sort by date
stock_joined_sorted = stock_joined_renamed.sort_values(by="date")
stock_joined_sorted.head()

,ticker,date,permno,price,monthly_return
0,MSFT,2024-01-31,10107,397.57999,0.057281
24,NVDA,2024-01-31,86580,615.27002,0.242418
12,AAPL,2024-01-31,14593,184.39999,-0.042227
1,MSFT,2024-02-29,10107,413.64001,0.042281
25,NVDA,2024-02-29,86580,791.12,0.285809


In [26]:
# Extract year/month/day
stock_joined_sorted["year"] = stock_joined_sorted["date"].dt.year
stock_joined_sorted["month"] = stock_joined_sorted["date"].dt.month
stock_joined_sorted["day"] = stock_joined_sorted["date"].dt.day
stock_joined_sorted.head()


,ticker,date,permno,price,monthly_return,year,month,day
0,MSFT,2024-01-31,10107,397.57999,0.057281,2024,1,31
24,NVDA,2024-01-31,86580,615.27002,0.242418,2024,1,31
12,AAPL,2024-01-31,14593,184.39999,-0.042227,2024,1,31
1,MSFT,2024-02-29,10107,413.64001,0.042281,2024,2,29
25,NVDA,2024-02-29,86580,791.12,0.285809,2024,2,29


In [27]:
#5. Descriptive Analysis

# Top returns
stock_by_return = stock_joined_sorted[stock_joined_sorted["monthly_return"].notna()].sort_values(by="monthly_return", ascending=False)
stock_by_return.head(10)

,ticker,date,permno,price,monthly_return,year,month,day
25,NVDA,2024-02-29,86580,791.12,0.285809,2024,2,29
28,NVDA,2024-05-31,86580,1096.32996,0.268871,2024,5,31
24,NVDA,2024-01-31,86580,615.27002,0.242418,2024,1,31
26,NVDA,2024-03-28,86580,903.56,0.142178,2024,3,28
16,AAPL,2024-05-31,14593,192.25,0.130159,2024,5,31
29,NVDA,2024-06-28,86580,123.54,0.126942,2024,6,28
17,AAPL,2024-06-28,14593,210.62,0.095553,2024,6,28
33,NVDA,2024-10-31,86580,132.75999,0.093215,2024,10,31
5,MSFT,2024-06-28,10107,446.95001,0.076651,2024,6,28
4,MSFT,2024-05-31,10107,415.13,0.068194,2024,5,31


In [28]:
# Summary statistics by ticker
summary_stats = stock_joined_sorted.groupby("ticker")["monthly_return"].agg(
    mean_return="mean",
    std_risk="std",
    min_return="min",
    max_return="max",
    count="count"
).round(4)

summary_stats



,mean_return,std_risk,min_return,max_return,count
ticker,,,,,
AAPL,0.024,0.0563,-0.0513,0.1302,12
MSFT,0.0114,0.0523,-0.0746,0.0767,12
NVDA,0.0928,0.1215,-0.0528,0.2858,12


In [30]:
db.close()

In [31]:
stock_joined_sorted.to_csv("stock_2022_data.csv", index=False)